---
authors:
  - edesz
date: 2026-04-08
---

# Profiling At-Risk Customers


In [ ]:
import os
from pathlib import Path

import altair as alt
import boto3
import numpy as np
import pandas as pd
from dotenv import load_dotenv

Below are `altair` plotting settings


In [ ]:
_ = alt.data_transformers.enable("vegafusion")
_ = alt.renderers.set_embed_options(actions=False)

In [ ]:
PROJ_ROOT = Path.cwd().parent

In [ ]:
assert load_dotenv(dotenv_path=PROJ_ROOT.parent / ".env")

In [ ]:
import cc_churn.costs as costs
import cc_churn.viz_altair as vzu
import r2.io_utils as r2io
from utils.df_utils import show_df

## About

In this notebook, we will extract profiles about the customers perdicted to be at risk of canceling their credit card services at the bank.

Profiles will be created by characterizing at-risk and safe (not at-risk) customers using categorical and ordinal features and separately by numerical features. Based on the findings, we will generate recommendations for the client to follow in order to target the at-risk customers.

:::{note}
### Outputs

Charts will be saved as `.html` files in `reports/figures`.
:::

## User Inputs

In [ ]:
columns_features_pred_proba = [
    "clientnum",
    "income_category",
    "education_level",
    "marital_status",
    "dependent_count",
    "customer_age",
    "gender",
    "card_category",
    "months_on_book",
    "num_products",
    "months_inactive_12_mon",
    "contacts_count_12_mon",
    "credit_limit",
    "total_revolv_bal",
    "avg_open_to_buy",
    "total_amt_chng_q4_q1",
    "total_trans_amt",
    "total_trans_ct",
    "total_ct_chng_q4_q1",
    "avg_utilization_ratio",
    "y_pred_proba",
    "y_pred",
    "best_decision_threshold",
    "is_churned",
]

# costs
# # revenue from transactions (bank earns #% of transaction volume)
interchange_rate = 0.02
# # revenue from revolving balance (~20% interest)
apr = 0.18
# # fee revenue from credit card exposure (modeled from card type)
card_fees = {"Blue": 0, "Silver": 50, "Gold": 100, "Platinum": 200}
tenure_years = 3
discount = 0.9
# # percentage of churners who can be convinced to stay (i.e. success rate
# # of saving a churning customer)
success_rate = 0.40
# # cost of intervention to get a single customer to not churn (discounts,
# # call center time, retention offers, etc.)
intervention_cost = 50
# # maximum number of customers that can be targeted based on client's budget
num_customers_max = 100

# predictions prefix
r2_key_pred = "2026-04-13/all_predictions__"

In [ ]:
reports_folder = PROJ_ROOT / "reports"
figures_folder = reports_folder / "figures"

account_id = os.getenv("ACCOUNT_ID")
access_key_id = os.getenv("ACCESS_KEY_ID")
secret_access_key = os.getenv("SECRET_ACCESS_KEY")
bucket_name = os.getenv("BUCKET_NAME")

s3_client = boto3.client(
    "s3",
    endpoint_url=f"https://{account_id}.r2.cloudflarestorage.com",
    aws_access_key_id=access_key_id,
    aws_secret_access_key=secret_access_key,
    region_name="auto",
)

# costs
multiplier = (1 - discount**tenure_years) / (1 - discount)

## Load Data

Load churn predictions for all customers

In [ ]:
%%time
df_all_pred = r2io.pandas_read_filtered_parquets_r2(
    s3_client,
    bucket_name,
    r2_key_pred,
    columns_features_pred_proba
).assign(is_at_risk=lambda df: df["y_pred"])
print(f"Got {len(df_all_pred):,} rows of all available data")
_ = show_df(df_all_pred)
with pd.option_context('display.max_columns', None):
    display(df_all_pred.head(1))

Use model predictions of all data to extract best decision threshold


In [ ]:
best_decision_threshold = (
    df_all_pred["best_decision_threshold"].head(1).squeeze()
)

Calculate business metrics for targeting all customers (those predicted to churn and those not predicted to do so)


In [ ]:
%%time
df_business_metrics, _, _ = costs.get_cost(
    df_all_pred,
    best_decision_threshold,
    interchange_rate,
    apr,
    card_fees,
    multiplier,
    success_rate,
    intervention_cost,
    False,
)
_ = show_df(df_business_metrics)
with pd.option_context('display.max_columns', None, 'display.max_colwidth', None):
    display(df_business_metrics.head(1))

## Exploratory Data Analysis (Post-Hoc)

### Characteristics of At-Risk Customers

Get a summary of the categorical and ordinal characteristics for all customers who are predicted to be and not to be at risk of churning

In [ ]:
%%time
df_categorical_risk_attributes = (
    pd.concat(
        [
            (
              df_business_metrics
              .groupby([f], observed=True)
              .agg({"is_at_risk": ["sum", 'count']})
              .set_axis(['number_at_risk', 'total_customers'], axis=1)
              .assign(
                  feature=f,
                  fraction_of_customers_at_risk=lambda df: (
                      df['number_at_risk'].div(df['total_customers']).mul(100)
                  ),
              )
              .reset_index()
              .rename(columns={f: "category"})
          )
            for f in [
                'card_category',
                'dependent_count',
                'education_level',
                'gender',
                'income_category',
                'marital_status',
            ]
        ]
    )
    .sort_values(
        by=['feature', 'fraction_of_customers_at_risk'],
        ascending=[True, False],
    )
    .set_index(['feature', 'category'])
    .reset_index()
)
df_categorical_risk_attributes

**Observations**

From the `fraction_of_customers_at_risk` column across various categorical and ordinal features, we can make the following observations

1. Uniform Risk Distribution
   - The risk of churn is remarkably consistent across most sub-categories for all such non-numerical features. As an example, whether looking at `income_category`, `education_level`, or `marital_status`, the fraction of at-risk customers almost entirely falls between *14% and 18%*. This suggests that demographic factors alone are not strong differentiators for churn in this specific dataset.
2. Small Outliers in Education and Card Type
   - While the distribution is mostly flat, a few groups show slightly higher risk
     - Doctorate holders have the highest risk in the education category at ~20.8%.
     - Platinum cardholders show a risk of 20%, though the sample size for this group is very small (only 20 total customers).
3. Gender Consistency
   - There is only a slight difference between genders, with *Females (~16.9%)* being marginally more likely to be classified as at-risk than *Males (~15.1%)*.
4. Implication from ML Model Development
   - Because the risk is so evenly spread across these categorical features, it implies that the ML model is likely relying much more heavily on *behavioral features* (such as transaction counts, revolving balances, and contact frequency) rather than demographic profiles to make its predictions. This is not surprising and is infact expected since we only used numerical features in the best ML model. Churn appears to be a result of *how* the customer uses the service, rather than *who* the customer is.

In summary, this suggests that churn is driven more by behavior, like transaction activity, rather than by fixed attributes like income or education.

Next, we'll get a summary of the numerical characteristics for all customers who are predicted to be and not to be at risk of canceling their credit card services at the bank


In [ ]:
%%time
df_summary_stats = (
    df_business_metrics.groupby('is_at_risk').agg(
        {
            'clientnum': 'count',
            'customer_age': 'mean',
            'months_on_book': 'mean',
            'num_products': 'mean',
            'months_inactive_12_mon': 'mean',
            'contacts_count_12_mon': 'mean',
            'credit_limit': "mean",
            'total_revolv_bal': 'mean',
            'avg_open_to_buy': 'mean',
            'total_amt_chng_q4_q1': 'mean',
            'total_trans_amt': 'mean',
            'total_ct_chng_q4_q1': 'mean',
            'total_trans_ct': 'mean',
            'avg_utilization_ratio': 'mean',
        }
    )
    .rename(
        index={True: 'At-Risk Customers', False: 'Stable Customers'},
        columns={'clientnum': 'num_customers'},
    )
    .transpose()
    .round(3)
    .reset_index()
    .rename(columns={"index": "Column"})
)
df_summary_stats

**Observations**

1. Below are the columns to focus on in charts, since they show a difference between the at-risk customers (those at risk of churning) and stable customers (those not at risk of churning)
   - `contacts_count_12_mon`
   - `total_revolv_bal`
   - `avg_utilization_ratio`
   - `total_trans_ct`

### Customer Profiling

#### Number of Credit Card Transactions

In [ ]:
chart = vzu.plot_grouped_overlapping_altair_histogram(
    df=df_business_metrics,
    num_bins=75,
    xvar="total_trans_ct:Q",
    xtitle="Total Transaction Count",
    ytitle="Number of Customers",
    color_by_col="is_at_risk:N",
    legend_title="At Risk",
    ptitle=alt.TitleParams(
        text=(
            "At-Risk Customers Have ~40 Transactions versus ~70 for Stable "
            "Customers"
        ),
        fontSize=18,
        font="Arial",
        anchor="start",
        orient="top",
        dx=50,
        offset=10,
    ),
    y_scale="linear",
    tooltip=["total_trans_ct"],
    scale_params=dict(domain=[True, False], range=["red", "lightgrey"]),
    # save_params=dict(
    #     fpath=figures_folder / "fig_31_profile_total_trans_ct_histogram.html"
    # ),
    fig_size=dict(width=700, height=350),
)
chart

At-risk customers are primarily focused in the *lower transaction count range* (35-50 transactions). This compares to stable customers who are in the 25-55 transactions range (lower) or in the 65-90 transactions (higher) range. This suggests that a drop in transaction frequency is an indicator of churn.

#### Credit Usage

In [ ]:
chart = vzu.plot_altair_scatter_chart(
    df=df_business_metrics,
    xvar="total_revolv_bal:Q",
    yvar="avg_utilization_ratio:Q",
    xtitle="Total Revolving Balance ($)",
    ytitle="Avg Utilization Ratio",
    color_by_col="is_at_risk:N",
    legend_title="At Risk",
    ptitle=alt.TitleParams(
        text="At-Risk Customers Tend to have a Low Card Usage and a Low Balance",
        fontSize=18,
        font="Arial",
        anchor="start",
        orient="top",
        dx=50,
        offset=10,
    ),
    xscale="linear",
    yscale="linear",
    scale_params=dict(domain=[True, False], range=["red", "lightgrey"]),
    # save_params=dict(
    #     fpath=figures_folder
    #     / ("fig_32_profile_tot_revol_balvs_avg_util_ratio.html")
    # ),
    fig_size=dict(width=700, height=350),
)
chart

At-risk customers (yellow) often show nearly zero revolving balances and very low utilization ratios. This *Dormant* behavior indicates they are no longer relying on the card for their daily financial needs.

#### Customer Engagement

In [ ]:
chart = vzu.plot_grouped_overlapping_altair_bar_chart(
    df=df_business_metrics,
    xvar="contacts_count_12_mon:O",
    xtitle="Number of Contacts (Last 12 Months)",
    ytitle="Number of Customers",
    color_by_col="is_at_risk:N",
    legend_title="At Risk",
    ptitle=alt.TitleParams(
        text=(
            "At-Risk Customers are more Likely to have at least 1 Contact in the "
            "Last Year"
        ),
        fontSize=18,
        font="Arial",
        anchor="start",
        orient="top",
        dx=50,
        offset=10,
    ),
    scale_params=dict(domain=[False, True], range=["lightgrey", "darkred"]),
    y_scale="linear",
    # save_params=dict(
    #     fpath=figures_folder
    #     / ("fig_33_profile_contacts_count_12_mon_histogram.html"),
    # ),
    fig_size=dict(width=700, height=350),
)
chart

**Observations**

1. As the count moves from 0 to 1-6, while the absolute number of customers decreases at very high contact counts, the proportion of red to grey increases.
2. For 0 contacts, stable customers (grey) outnumber the at-risk customers. However, at 1+ contacts, the red bars become a much larger share of the total bar height for each category. In the categories for 2, 3, 4, 5, and 6 contacts, the red portion of the bar represents a much larger percentage of that specific group compared to the lower contact groups. This indicates that these customers have a significantly higher likelihood of being at risk to cancel their credit card services at the bank. This suggests that high contact frequency is a strong predictor of churn risk. This suggests that as a customer is forced to contact the bank more frequently, likely due to unresolved issues, the probability that the ML model classifies them as at-risk increases strongly.

In summary, customers with *1 or more contacts in the last year* have a significantly higher likelihood of being classified as at-risk. *Medium to high contact volume without resolution* likely drives dissatisfaction. This suggests that unresolved issues are an important driver of churn.

#### Profiles of At-Risk Customers

1. Dormancy
   - At-risk customers typically have significantly lower transaction counts (averaging 44 transactions compared to approximately 70 transactions for stable customers from `total_trans_ct`) and much lower revolving balances (approximately 617 dollars versus approximately 1,268 dollars, see `total_revolv_bal`). This suggests they have stopped using their credit card from this bank as their primary payment method.
2. Inactivity
   - At-risk customers show higher periods of inactivity (their `months_inactive_12_mon` is approximately 2.7 versus only 2.3 for stable customers) and lower utilization ratios (`avg_utilization_ratio` is ~0.15 versus ~0.30). This suggests a loss of engagement before the actual cancellation.
3. High Friction
   - Customers with 1 or more contacts in the last year are much more likely to be at risk (see the bar chart of `contacts_count_12_mon`). This indicates that unresolved issues are a driver of churn.

#### Recommendations Based on Identified Customer Profiles

Below are the targeting recommendations per customer profile

1. Dormant Users (Low Activity/Low Balance)
   - launch a *re-activation* campaign with cashback incentives or point multipliers for the next few transactions to help them rebuild the usage habit again
2. Frustrated Customers (High Contacts)
   - perform direct outreach from a *Premier Customer Support* representative to resolve outstanding issues and offer a one-time fee waiver
3. The Low-Value Holder (Single Product)
   - offer incentives to bundle credit services with other bank products (e.g. savings account linked rewards) to increase switching costs to competitor

## Conclusion

### Summary of Findings

In this notebook, we profiled the customers predicted to be at risk of canceling their credit card services. The analysis revealed three primary customer profiles and key behavioral indicators of churn

1. Transactional Dormancy
   - The at-risk customers show a significant drop in engagement, averaging only 44 transactions compared to 69 for stable customers. Their revolving balances (`total_revolv_bal`) are also roughly 50% lower (617 dollars versus 1,268 dollars), indicating the card is no longer their primary payment method.
2. Increased Friction
   - A high number of customer service contacts is a strong leading indicator of churn. Customers with 1 or more contacts in the last 12 months are significantly more likely to be classified as at-risk, likely due to unresolved service issues.
3. Inactivity and Low Utilization
   - At-risk customers exhibit longer periods of inactivity and maintain a utilization ratio of only 15% (vs. 30% for stable customers), signaling a gradual withdrawal from the bank's services.
4. Demographic Uniformity
   - Churn risk is relatively uniform across demographic categories (income, education, marital status). It falls in therange between 14% and 18%. This confirms that churn is driven by behavioral usage patterns rather than fixed customer attributes. This is in line with the type of features (numerical only) used in ML model development.

### Recommendations for Strategic Targeting

In order to target these at-risk customers, we recommend the bank should implement the following strategies

1. Re-activation Campaigns
   - target *Dormant* users with cashback or point-multiplier incentives to rebuild card usage habits
2. Priority Resolution
   - deploy a *Premier Support* team to reach out to *Frustrated* customers (high contact counts) to resolve pending issues and offer retention waivers
3. Product Bundling
   - increase the switching costs for low-utilization users by offering incentives to link their credit cards with other high-value bank products like savings accounts